In [9]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [10]:
from openai import OpenAI
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini =OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)
response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])
print(response.choices[0].message.content)


A fun fact for you: **Sea otters hold hands when they sleep so they don’t drift away from each other.**

They often sleep in groups called a "raft," and they will sometimes wrap themselves in giant kelp to act as an anchor, keeping them securely in one spot while they snooze!


In [11]:
response=gemini.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Oman!"},
    {"role": "assistant", "content": "Hi Oman! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]
)
print(response.choices[0].message.content)

Your name is Oman!


In [12]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

print(tokens)

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148]


In [13]:
import json
from IPython.display import Markdown, display, update_display
from scrapper import fetch_website_contents,fetch_website_links

In [14]:
links = fetch_website_links("https://edwarddonner.com")
print(links)

['#wp--skip-link--target', 'https://edwarddonner.com/avatar/', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/proficient/', 'https://edwarddonner.com/connect-four/', 'https://edwarddonner.com/outsmart/', 'https://edwarddonner.com/about-me-and-about-nebula/', 'https://edwarddonner.com/posts/', 'https://edwarddonner.com/', 'https://news.ycombinator.com', 'https://nebula.io/?utm_source=ed&utm_medium=referral', 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/avatar/', 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/', 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/', 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/', 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/', 'https://edwarddonner.com/

In [15]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [16]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [17]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite", 
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [18]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [24]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling gemini-3.1-flash-lite")
    response = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [25]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-3.1-flash-lite
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'professional profile',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'company website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'curriculum', 'url': 'https://edwarddonner.com/curriculum/'}]}

In [26]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

In [28]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [29]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
MiniMaxAI/MiniMax-H3
Updated
about 22 hours ago
•
47.5k
•
3.47k
meta-models/Muse-Glimmer-30B
Updated
about 8 hours ago
•
867
deepseek-ai/DeepSeek-V4-Flash-0731
Updated
10 days ago
•
954k
•
3.09k
Comfy-Org/MiniMax-H3
Updated
2 days ago
•
6.01M
•
1.17k
larryvrh/MiniMax-H3-Turb

In [31]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [32]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [33]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 9 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nMiniMaxAI/MiniMax-H3\nUpdated\nabout 22 hours ago\n•\n47.5k\n•\n3.47k\nmeta-models/Muse-Glimmer-30B\nUpdated\nabout 

In [37]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-3.6-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [38]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


# Welcome to Hugging Face
*The AI community building the future.*

---

## About Hugging Face

Hugging Face is the leading collaboration platform for the global machine learning community. Known as the "Home of Machine Learning," Hugging Face provides the tools, storage, and infrastructure necessary for developers, researchers, and organizations to create, discover, and collaborate on AI better and faster. 

By prioritizing open source, accessibility, and community innovation, Hugging Face serves as the central hub driving the global AI ecosystem forward.

---

## What We Offer

### The Open Collaboration Hub
* **2 Million+ AI Models:** Discover, fine-tune, and collaborate on state-of-the-art open-weights models across text, image, video, audio, and multimodal tasks.
* **500k+ Datasets:** Host, manage, and explore high-quality training and evaluation datasets.
* **1 Million+ Applications (Spaces):** Showcase and host interactive machine learning applications and live demos directly on the platform.

### Solutions for Enterprise & Developers
Hugging Face delivers robust enterprise infrastructure designed for teams building production-ready AI applications:

* **Inference Endpoints & Providers:** Deploy models to secure, scalable compute infrastructure with a single click.
* **Storage Buckets:** High-performance cloud storage tailored specifically for large datasets and model weights.
* **Enterprise Support & Pro Plans:** Advanced team collaboration tools, custom enterprise support, and specialized hardware access for high-throughput AI tasks.
* **HuggingChat & Developer Ecosystem:** Open-source conversational AI interfaces, popular libraries (such as `transformers`), and learning resources including Hugging Face Fundamentals.

---

## Customers & Ecosystem

From individual AI enthusiasts and academic researchers to pioneering AI startups and Fortune 500 enterprises, millions of users rely on Hugging Face to host, evaluate, and scale their machine learning workflows. Organizations choose Hugging Face to move faster, stay cloud-agnostic, and leverage the collective output of the open-source community.

---

## Culture & Careers

### Our Culture: Open, Fast-Moving, and Community-Driven
At Hugging Face, open source is not just a distribution model—it is a core philosophy. We believe that open collaboration leads to safer, more accessible, and more powerful AI for everyone.

* **Frontier Innovation:** Our team works on the cutting edge of AI, from agentic systems and model linting to multimodal foundations and decentralized compute dynamics.
* **Global Community Impact:** Our team members directly shape how millions of developers build software daily through tools, articles, daily paper roundups, and active forums.
* **Autonomy & Empowerment:** With a fast-growing team of over 180+ passionate professionals, we value autonomy, rapid execution, and open communication.

### Join the Team
We are always looking for curious, talented engineers, researchers, and creators who want to build the future of machine learning. If you are passionate about open source, modern AI infrastructure, and empowering a global developer community, Hugging Face is the place for you.

---

## Connect With Us

* **Website:** huggingface.co
* **Community:** Join our active channels on Discord, GitHub, and the Hugging Face Forum to start collaborating today.

In [40]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model="gemini-3.6-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [41]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 7 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the leading platform where the machine learning community collaborates on models, datasets, and applications. Widely recognized as the home of machine learning, Hugging Face empowers developers, researchers, and enterprises to create, discover, and deploy open-source AI with unprecedented speed and efficiency.

---

## What We Offer (For Customers & Enterprises)

Hugging Face delivers a comprehensive platform designed for every stage of the machine learning lifecycle:

* **Unmatched ML Ecosystem**: Discover and build with over **2 million models**, **500,000+ datasets**, and **1 million+ applications** (Spaces).
* **Collaboration Platform**: Securely host and collaborate on unlimited public or private models, datasets, and interactive demos.
* **Storage & Infrastructure Solutions**: Utilize dedicated features such as Storage Buckets, zero-GPU Spaces, and seamless integrations for AI agents.
* **Enterprise & Production Services**:
  * **Inference Endpoints & Providers**: Scale ML models directly into production with fast, managed infrastructure.
  * **Team & Enterprise Plans**: Scale collaboration with organizational controls, dedicated enterprise support, and private hosting.
  * **Hugging Face PRO**: Unlock boosted compute and power-user capabilities for individual creators and developers.

---

## Community & Research Leadership (For Investors)

Hugging Face is at the heart of the global open-source AI momentum:

* **Driving Open Science**: We host research, track trending daily papers, and publish thought leadership on open-source ecosystems, agentic reinforcement learning, and compute landscapes.
* **Developer Ecosystem**: From educational tracks like *Hugging Face Fundamentals* to active community forums, Discord channels, and GitHub integrations, Hugging Face serves as the central hub for global AI innovation.
* **Agentic & Developer Tooling**: Continuously shipping state-of-the-art tooling, including linter utilities, coding agents, and multi-modal application frameworks.

---

## Culture & Careers (For Prospective Recruits)

With a growing global team of over 180+ core team members, Hugging Face offers an environment built on openness, impact, and continuous learning.

### Why Join Hugging Face?
* **Open-Source Ethos**: We believe that the future of technology belongs to open science, shared tools, and transparent collaboration.
* **High Impact & Autonomy**: Work on tools used by millions of developers and the world's leading organizations every single day.
* **Innovative Environment**: Build at the cutting edge of AI agents, open models, compute optimizations, and open-source infrastructure.

*Explore current open positions on our Careers page and help us build the future of AI.*